# 05 — Evaluation, inventory policy, and business takeaways

Load artifacts from `python -m src.pipeline` (run that once if `results/` is empty).


In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"
VIZ = RESULTS / "visualizations"

perf = pd.read_csv(RESULTS / "model_performance.csv")
clf = pd.read_csv(RESULTS / "classification_performance.csv")
inv = pd.read_csv(RESULTS / "inventory_optimization.csv")
insights = pd.read_json(RESULTS / "business_insights.json", typ="series")
perf


## Regression leaderboard


In [ ]:
display(perf.sort_values("WAPE"))
for name in ["regression_wape.png", "regression_mape.png", "scatter_vendor.png", "scatter_operational.png", "pred_vs_actual_vendor.png", "feature_importance.png"]:
    path = VIZ / name
    if path.exists():
        display(Markdown(f"**{name}**"))
        display(Image(str(path)))


## Classification


In [ ]:
display(clf.sort_values("accuracy", ascending=False))
display(Image(str(VIZ / "confusion_matrix.png")))
display(Image(str(VIZ / "classification_accuracy.png")))


## Approach 3 — inventory optimization

Assumptions (also in `src/config.py`):

- Holding cost = 25% of price per year, charged daily on on-hand units
- Stockout cost = 45% of price per lost unit (lost margin)
- Lead time = 1 day (the dataset is a daily on-hand snapshot)
- Safety stock = z × residual std × √lead_time
- z is chosen on the **train** period to minimize total cost


In [ ]:
display(inv)
display(Image(str(VIZ / "inventory_cost_comparison.png")))
pd.read_csv(RESULTS / "safety_stock_grid.csv")


## Clustering (segmentation check)


In [ ]:
skus = pd.read_csv(RESULTS / "sku_segments.csv")
display(skus.groupby("segment")[["mean_sales", "mean_price", "mean_inventory", "turnover", "revenue"]].mean())
display(Image(str(VIZ / "sku_clusters.png")))
print("Because SKU means are almost identical in this file, clusters will be weak. That is a finding, not a failure.")


## Takeaways

1. **Forecasting:** Use the vendor `Demand Forecast` as the production signal (WAPE ≈ 6%). Operational-only ML cannot honestly beat ~R² 0.35 on this file.
2. **Classification:** Vendor-track models exceed 75% accuracy because they can read the existing forecast. Operational-only classifiers sit near 55–60%.
3. **Inventory:** Current on-hand is far above realized daily demand. A forecast + safety-stock target cuts holding cost; watch the stockout rate as z changes.
4. **Pricing / weather / promotions:** No material mean shift in this dataset. Do not recommend discount policy changes from these columns.
5. **Metrics:** Prefer WAPE and MAE for reporting. Quote MAPE only on days with demand ≥ 10, and say so.
